In [29]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.3.5
Pandas version: 3.0.0


## 1. Load & Inspect

In [30]:
df = pd.read_csv('messy_sales.csv')

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nFirst 5 rows:")
df.head()

Shape: (10, 8)

Dtypes:
 order_id           int64
customer_name        str
age                  str
city                 str
product              str
quantity         float64
price            float64
order_date           str
dtype: object

First 5 rows:


,order_id,customer_name,age,city,product,quantity,price,order_date
0,1001,Alice,25,New York,Laptop,2.0,1200.0,2024-01-15
1,1002,Bob,NaN,London,Phone,1.0,800.0,2024-01-16
2,1003,Charlie,35,NaN,Tablet,3.0,450.0,2024-01-17
3,1004,Alice,25,New York,Laptop,2.0,1200.0,2024-01-15
4,1005,Diana,28,Paris,Phone,NaN,700.0,2024-01-18


In [31]:
# Info + missing values + duplicates
print(df.info())
print("\nMissing values per column:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nBasic stats:\n", df.describe(include='all'))

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       10 non-null     int64  
 1   customer_name  10 non-null     str    
 2   age            8 non-null      str    
 3   city           9 non-null      str    
 4   product        10 non-null     str    
 5   quantity       9 non-null      float64
 6   price          9 non-null      float64
 7   order_date     10 non-null     str    
dtypes: float64(2), int64(1), str(5)
memory usage: 772.0 bytes
None

Missing values per column:
 order_id         0
customer_name    0
age              2
city             1
product          0
quantity         1
price            1
order_date       0
dtype: int64

Duplicate rows: 0

Basic stats:
           order_id customer_name  age      city product  quantity  \
count     10.00000            10    8         9      10       9.0   
unique         NaN             9    

## 2. Clean the Data

### Issues found:
1. Missing values in age, city, quantity, price
2. Duplicate row (order 1004 duplicates 1001)
3. `age` column contains "thirty" → should be numeric
4. `order_date` should be datetime

In [32]:
# age → numeric (coerce errors to NaN first)
df['age'] = pd.to_numeric(df['age'], errors='coerce')

# order_date → datetime
df['order_date'] = pd.to_datetime(df['order_date'])

print(df.dtypes)

order_id                  int64
customer_name               str
age                     float64
city                        str
product                     str
quantity                float64
price                   float64
order_date       datetime64[us]
dtype: object


In [33]:
df = df.drop_duplicates()
print("Shape after dedup:", df.shape)

Shape after dedup: (10, 8)


In [34]:
# Strategy:
# - age: fill with median (robust to outliers)
# - city: fill with mode (categorical)
# - quantity: fill with median
# - price: fill with median (could also drop, but we don't want to lose rows)

df['age'] = df['age'].fillna(df['age'].median())
df['city'] = df['city'].fillna(df['city'].mode()[0])
df['quantity'] = df['quantity'].fillna(df['quantity'].median())
df['price'] = df['price'].fillna(df['price'].median())

print("Missing values remaining:\n", df.isnull().sum())

Missing values remaining:
 order_id         0
customer_name    0
age              0
city             0
product          0
quantity         0
price            0
order_date       0
dtype: int64


## 3. Feature Engineering

Add a `total_amount` column = quantity × price.

In [35]:
df['total_amount'] = df['quantity'] * df['price']
df.head()

,order_id,customer_name,age,city,product,quantity,price,order_date,total_amount
0,1001,Alice,25.0,New York,Laptop,2.0,1200.0,2024-01-15,2400.0
1,1002,Bob,29.0,London,Phone,1.0,800.0,2024-01-16,800.0
2,1003,Charlie,35.0,New York,Tablet,3.0,450.0,2024-01-17,1350.0
3,1004,Alice,25.0,New York,Laptop,2.0,1200.0,2024-01-15,2400.0
4,1005,Diana,28.0,Paris,Phone,2.0,700.0,2024-01-18,1400.0


## 4. GroupBy Practice

Answer these questions with code:
1. Total revenue per city
2. Average order value per product
3. Number of unique customers per city
4. Top 2 products by total quantity sold

In [36]:
# 1. Revenue per city
print("Revenue per city:\n", df.groupby('city')['total_amount'].sum().sort_values(ascending=False))

# 2. Avg order value per product
print("\nAvg order value per product:\n", df.groupby('product')['total_amount'].mean().round(2))

# 3. Unique customers per city
print("\nUnique customers per city:\n", df.groupby('city')['customer_name'].nunique())

# 4. Top 2 products by quantity
print("\nTop 2 products by quantity:\n", df.groupby('product')['quantity'].sum().nlargest(2))

Revenue per city:
 city
New York    7050.0
Paris       4200.0
Berlin      1900.0
London      1700.0
Name: total_amount, dtype: float64

Avg order value per product:
 product
Laptop    1675.00
Phone     1666.67
Tablet    1050.00
Name: total_amount, dtype: float64

Unique customers per city:
 city
Berlin      2
London      2
New York    3
Paris       2
Name: customer_name, dtype: int64

Top 2 products by quantity:
 product
Phone     7.0
Tablet    7.0
Name: quantity, dtype: float64


## 5. Merge Practice

Create a small `customer_tier` DataFrame and merge it with df.

In [37]:
tier_df = pd.DataFrame({
    'customer_name': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 
                      'Frank', 'Grace', 'Henry', 'Ivy'],
    'tier': ['Gold', 'Silver', 'Gold', 'Bronze', 'Silver',
             'Bronze', 'Gold', 'Silver', 'Bronze']
})

df_merged = df.merge(tier_df, on='customer_name', how='left')
df_merged.head()

,order_id,customer_name,age,city,product,quantity,price,order_date,total_amount,tier
0,1001,Alice,25.0,New York,Laptop,2.0,1200.0,2024-01-15,2400.0,Gold
1,1002,Bob,29.0,London,Phone,1.0,800.0,2024-01-16,800.0,Silver
2,1003,Charlie,35.0,New York,Tablet,3.0,450.0,2024-01-17,1350.0,Gold
3,1004,Alice,25.0,New York,Laptop,2.0,1200.0,2024-01-15,2400.0,Gold
4,1005,Diana,28.0,Paris,Phone,2.0,700.0,2024-01-18,1400.0,Bronze


In [38]:
# Revenue per tier
print("Revenue per tier:\n", df_merged.groupby('tier')['total_amount'].sum().sort_values(ascending=False))

Revenue per tier:
 tier
Gold      8950.0
Bronze    3200.0
Silver    2700.0
Name: total_amount, dtype: float64


## 6. Pivot Table

In [39]:
pivot = df.pivot_table(
    values='total_amount',
    index='city',
    columns='product',
    aggfunc='sum',
    fill_value=0
)
pivot

product,Laptop,Phone,Tablet
city,,,
Berlin,1900.0,0.0,0.0
London,0.0,800.0,900.0
New York,4800.0,0.0,2250.0
Paris,0.0,4200.0,0.0


## 7. NumPy Practice

Now redo some pandas operations with pure NumPy to sharpen intuition.

In [40]:
# Convert to numpy arrays
quantities = df['quantity'].to_numpy()
prices = df['price'].to_numpy()

# Manual total
manual_total = quantities * prices
print("Manual total:", manual_total)

# Vectorized operations
print("Mean quantity:", np.mean(quantities))
print("Std price:", np.std(prices))
print("Max total:", np.max(manual_total))

# Boolean masking
expensive = df[df['price'].to_numpy() > 700]
print("\nExpensive orders:\n", expensive[['order_id', 'product', 'price']])

Manual total: [2400.  800. 1350. 2400. 1400.  700.  900. 2800. 1200.  900.]
Mean quantity: 2.0
Std price: 295.8462438497403
Max total: 2800.0

Expensive orders:
    order_id product   price
0      1001  Laptop  1200.0
1      1002   Phone   800.0
3      1004  Laptop  1200.0
8      1009  Laptop  1200.0


# Day 1 — NumPy & Pandas Revision

## Goal
Re-activate NumPy and Pandas fundamentals by cleaning a deliberately 
messy dataset.

## Topics Covered
- Loading & inspecting data
- Handling missing values, duplicates, wrong dtypes
- Feature engineering
- GroupBy, Merge, Pivot Table
- NumPy vectorization

## Dataset
`data/messy_sales.csv` — contains:
- Missing values
- Duplicate rows
- Mixed types (e.g., "thirty" in age column)
- Datetime parsing issues

## Key Takeaways
- Always inspect before cleaning
- Justify every imputation strategy
- Vectorized ops > loops